In [1]:
# %pip install -q tensorflow-datasets matplotlib
# %pip install -q importlib_resources

In [2]:
import os, time, json, platform, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

SEED = 42
CALIBRATION_IMAGES = 100
TEST_IMAGES = 500
WARMUP_RUNS = 10
TIMING_REPEATS = 1
IMAGE_SIZE = (224, 224)
OUTPUT_DIR = "mobilenetv2_tflite_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
# Loading and processing the dataset - ImageNet-V2

total = CALIBRATION_IMAGES + TEST_IMAGES
ds = tfds.load("imagenet_v2", split=f"test[:{total}]",
               as_supervised=True, shuffle_files=False)

images, labels = [], []
for image, label in tfds.as_numpy(ds):
    image = tf.image.resize(image, IMAGE_SIZE).numpy().astype(np.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    images.append(image)
    labels.append(int(label))

images = np.stack(images).astype(np.float32)
labels = np.asarray(labels, dtype=np.int64)

calibration_images = images[:CALIBRATION_IMAGES]
test_images = images[CALIBRATION_IMAGES:CALIBRATION_IMAGES + TEST_IMAGES]
test_labels = labels[CALIBRATION_IMAGES:CALIBRATION_IMAGES + TEST_IMAGES]

In [4]:
# Loading the model

model = tf.keras.applications.MobileNetV2(
    weights="imagenet", include_top=True, input_shape=(224, 224, 3)
)

In [5]:
# Keras -> SavedModel format ->TensorflowLite (FP32)

saved_model_dir = os.path.join(
    OUTPUT_DIR, "mobilenetv2_savedmodel"
)

# export keras model to a SavedModel format first
model.export(saved_model_dir)

# convert the SavedModel to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_saved_model(
    saved_model_dir
)

fp32_model = converter.convert()

# save the converted model
fp32_path = os.path.join(
    OUTPUT_DIR, "mobilenetv2_fp32.tflite"
)

with open(fp32_path, "wb") as f:
    f.write(fp32_model)


Saved artifact at 'mobilenetv2_tflite_experiment/mobilenetv2_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1000), dtype=tf.float32, name=None)
Captures:
  135411671900688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671901456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671901264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671901648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671899536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671901840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671900880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671902608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671901072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135411671900496: TensorSpec(shape=(), dtype=t

In [6]:

# SavedModel format -> TensorflowLite (FP32) -> Quantize -> INT8

saved_model_dir = os.path.join(
    OUTPUT_DIR, "mobilenetv2_savedmodel"
)

# Creating int8 converter from SavedModel
converter = tf.lite.TFLiteConverter.from_saved_model(
    saved_model_dir
)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

# this function is useful for the calibration of the calibration images for quantization
def representative_dataset():
    for image in calibration_images:
        image = np.asarray(image, dtype=np.float32)

        # Ensure each image has a batch dimension
        if image.ndim == 3:
            image = np.expand_dims(image, axis=0)

        yield [image]

converter.representative_dataset = representative_dataset

# Require integer-only operations
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

# carrying out conversion
int8_model = converter.convert()

# saving the int8 model
int8_path = os.path.join(
    OUTPUT_DIR, "mobilenetv2_int8.tflite"
)

with open(int8_path, "wb") as f:
    f.write(int8_model)


In [7]:
# function to prepare model to perform inference
# returns interpreter that executes model

def make_interpreter(model_bytes):
    interpreter = tf.lite.Interpreter(model_content=model_bytes, num_threads=1)
    interpreter.allocate_tensors()
    return interpreter

In [8]:
# takes in interpreter and image as arguments to perform prediction on one image
# returns the predicted class

def predict_one(interpreter, image):
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    x = np.expand_dims(image, axis=0)
    if np.issubdtype(inp["dtype"], np.integer):
        scale, zero = inp["quantization"]
        if scale == 0:
            raise ValueError("Input quantization scale is zero")
        x = np.clip(np.round(x / scale + zero),
                    np.iinfo(inp["dtype"]).min,
                    np.iinfo(inp["dtype"]).max).astype(inp["dtype"])
    else:
        x = x.astype(inp["dtype"])
    interpreter.set_tensor(inp["index"], x)
    interpreter.invoke()
    return int(np.argmax(interpreter.get_tensor(out["index"])[0]))

In [9]:
# function to benchmark and evaluate the models

def benchmark(name, model_bytes):
    interpreter = make_interpreter(model_bytes)
    for i in range(min(WARMUP_RUNS, len(test_images))):
        predict_one(interpreter, test_images[i])

    timings, correct = [], []
    for repeat in range(TIMING_REPEATS):
        for image, label in zip(test_images, test_labels):
            start = time.perf_counter()
            pred = predict_one(interpreter, image)
            timings.append(time.perf_counter() - start)
            correct.append(pred == int(label))

    timings = np.asarray(timings)
    accuracy = float(np.mean(correct))
    mean_seconds = float(np.mean(timings))
    return {
        "model": name,
        "images_measured": len(timings),
        "accuracy": accuracy,
        "mean_latency_ms": mean_seconds * 1000,
        "median_latency_ms": float(np.median(timings) * 1000),
        "p95_latency_ms": float(np.percentile(timings, 95) * 1000),
        "throughput_images_per_second": len(timings) / float(np.sum(timings)),
        "model_size_mib": len(model_bytes) / 1024**2,
        "vips": accuracy / mean_seconds if mean_seconds else float("nan"),
        "latencies_ms": (timings * 1000).tolist(),
        "correct": [bool(x) for x in correct]
    }

In [10]:
# accuracy -> number of correct predictions
# mean_latency_ms -> average time per prediction
# median_latency_ms -> median time per prediction
# p95_latency_ms -> 95th percentile latency
# throughput_images_per_second ->  number of predictions completed per uit time
# model_size_mib -> file size of teh model
# vips -> accuracy divided by mean latency in seconds
# latencies_ms -> individual recorded times
# correct -> correctness of individual predictions

## 8. Benchmark both models

In [11]:
fp32 = benchmark("MobileNetV2 FP32", fp32_model)
int8 = benchmark("MobileNetV2 INT8", int8_model)

def clean_result(r):
    return {k: v for k, v in r.items() if k not in ("latencies_ms", "correct")}

results = pd.DataFrame([clean_result(fp32), clean_result(int8)])
display(results.round(4))

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,images_measured,accuracy,mean_latency_ms,median_latency_ms,p95_latency_ms,throughput_images_per_second,model_size_mib,vips
0,MobileNetV2 FP32,500,0.574,16.1084,13.4101,22.9741,62.0796,13.3524,35.6337
1,MobileNetV2 INT8,500,0.572,21.7499,19.2117,29.4641,45.9773,3.8436,26.2990
